# 원신 캐릭터 정보 RAG — 모델 비교 (코랩용)

**하는 일**: rag_v4 프롬프트는 그대로 두고 **모델만 바꿔서** DEV 60문항으로 점수와 응답 시간을 비교한다.
결과는 JSON으로 저장하고, 보고서에서 **"왜 Qwen2.5-3B를 골랐는지"** 근거로 쓴다 (TODO(C) 9번).

**같게 두는 것 (공정한 비교)**
- 질문: DEV 60문항 (노트북 01 셀 7의 `build_dev_set()`과 같은 60문항, 학습용 질문에서 뽑음)
- 프롬프트: rag_v4 (`core/answer.py`의 `build_prompt`, 챗봇·노트북 01과 같은 글자)
- 채점: 노트북 01 셀 3과 같은 코드 (집합 F1)
- 생성 설정: 샘플링 끔, 최대 160토큰, `]`가 나오면 멈춤

**규칙**
- 모델은 **DEV로만** 고른다. 평가 115문항은 이 노트북에서 돌리지 않는다 (PRD 4.3).
- rag_v4 프롬프트는 Qwen2.5-3B로 고른 것이라 다른 모델에는 조금 불리할 수 있다. 결과에 함께 적는다.

**비교할 모델 (예)**

| MODEL_TAG | MODEL_ID | LOAD_4BIT | 왜 |
|---|---|---|---|
| qwen2.5-3b | Qwen/Qwen2.5-3B-Instruct | False | 기준. DEV 0.814가 그대로 나와야 이 노트북을 믿을 수 있다 |
| qwen2.5-1.5b | Qwen/Qwen2.5-1.5B-Instruct | False | 같은 계열, 작게 (빠름) |
| qwen2.5-7b-4bit | Qwen/Qwen2.5-7B-Instruct | True | 같은 계열, 크게 (T4 16GB라 4bit) |


**사용법**
1. 팀 드라이브 폴더 `Genshin Build Alchemist (팀)`을 우클릭 → 정리 → 바로가기 추가 → 내 드라이브 (처음 한 번만)
2. 코랩 왼쪽 열쇠 아이콘(보안 비밀)에 `GITHUB_TOKEN` 추가하고 노트북 액세스 켜기 (처음 한 번만)
3. `런타임 > 런타임 유형 변경 > T4 GPU` 선택
4. **처음에는 기준 모델(qwen2.5-3b)로** 셀 1~7을 차례로 실행해서 DEV macro F1이 0.814인지 확인한다
5. 다른 모델은 `런타임 > 세션 다시 시작` 후 셀 5의 `MODEL_ID`, `MODEL_TAG`, `LOAD_4BIT`만 바꿔 셀 1부터 다시 실행한다 (GPU 메모리 비우기)
6. 셀 6이 모델마다 결과 2개를 저장한다: `results/dev_rag_v4_{MODEL_TAG}.json`(후처리 없음), `..._post.json`(후처리 있음, 모델은 한 번만 돌림)
7. 셀 7이 모델별·후처리 유무별 결과를 표로 비교한다


In [ ]:
# 1. 설정: 저장소 코드(core/) 받기 + 드라이브 연결 + 데이터 불러오기
#    라우터·검색은 저장소의 core/ 코드를 그대로 쓴다 (노트북 평가와 실제 챗봇이 같은 코드로 동작하게)
import hashlib
import json
import os
import random
import re
import subprocess
import sys
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

REPO = "genshin-build-alchemist/genshin-build-alchemist"
REPO_BRANCH = "main"       # PR 브랜치를 시험할 때만 바꾼다 (예: "rag/router-tests")
IN_COLAB = "google.colab" in sys.modules


def github_token():
    """코랩 보안 비밀(왼쪽 열쇠 아이콘)의 GITHUB_TOKEN. 저장소가 공개면 없어도 된다."""
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return os.environ.get("GITHUB_TOKEN")


def run_git(args, token):
    """git 실행. 출력에 토큰이 찍히지 않게 가린다."""
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***") if token else r.stdout + r.stderr
    if r.returncode != 0:
        raise RuntimeError("git 실패:\n" + out + "\n저장소가 비공개면 코랩 보안 비밀에 GITHUB_TOKEN을 넣었는지 확인하세요 "
                           "(notebooks/README.md 참고).")


if IN_COLAB:
    REPO_DIR = Path("/content/genshin-build-alchemist")
    token = github_token()
    url = f"https://{'x-access-token:' + token + '@' if token else ''}github.com/{REPO}.git"
    if (REPO_DIR / ".git").exists():
        run_git(["-C", str(REPO_DIR), "fetch", "--depth", "1", url, REPO_BRANCH], token)
        run_git(["-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], token)
    else:
        run_git(["clone", "--depth", "1", "--branch", REPO_BRANCH, url, str(REPO_DIR)], token)
        run_git(["-C", str(REPO_DIR), "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], token)
else:   # 로컬: 이 노트북이 들어 있는 저장소를 쓴다 (GBA_REPO 환경변수로 지정 가능)
    REPO_DIR = Path(os.environ.get("GBA_REPO") or next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "core").is_dir()))
sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip() or "unknown"
print(f"저장소 코드: {REPO_DIR} ({REPO_BRANCH} @ {commit})")

from core.kb import load_kb                       # noqa: E402  (저장소 경로를 넣은 뒤에 가져온다)
from core.retrieval import character_line, retrieve as core_retrieve   # noqa: E402
from core.router import Router                    # noqa: E402

# 팀 드라이브 폴더 (공유받은 폴더를 '내 드라이브'에 바로가기로 추가해 두어야 이 경로가 보인다)
TEAM_DIR = Path("/content/drive/MyDrive/Genshin Build Alchemist (팀)")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
if TEAM_DIR.exists():
    HUMAN_PATH = TEAM_DIR / "data" / "human_questions.csv"     # 사람이 직접 만든 질문 (선택)
    RESULT_DIR = TEAM_DIR / "results"                         # 실험 결과 JSON 폴더 (대시보드가 읽는 곳)
    print(f"팀 드라이브 사용: {TEAM_DIR}")
else:
    HUMAN_PATH = Path("human_questions.csv")
    RESULT_DIR = Path("results")
    print("팀 드라이브 폴더가 없어서 현재 폴더를 씁니다. (결과가 세션 종료 시 사라지니 주의)")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_VERSION = "info_v2"          # 질문 생성 규칙을 바꾸면 이 이름도 바꾼다 (점수 비교 기준이 달라지므로)
SEED = 42
CODE_VERSION = f"{REPO_BRANCH}@{commit}"   # 결과 JSON의 config에 남긴다 (어느 코드로 돌렸는지)
# 같은 experiment_id로 다시 실행하면 드라이브의 결과 파일을 덮어쓴다. 남기고 싶으면 ID를 바꾼다.

# 데이터는 저장소의 data/ 를 쓴다 (드라이브 data/ 는 같은 파일의 사본)
KB = load_kb(REPO_DIR / "data")
with open(REPO_DIR / "data" / "characters_ko.json", encoding="utf-8") as f:
    raw_data = json.load(f)
characters = KB.characters
CHAR_BY_NAME = KB.char_by_name
print(f"캐릭터 {len(characters)}명 불러옴 (출처: {raw_data['source']}, {raw_data['commit']})")
for field in ("element", "weapon", "rarity", "region"):
    missing = [c["name"] for c in characters if not c[field]]
    print(f"  {field:<8} 값 {len({c[field] for c in characters if c[field]})}종, 정보 없음 {len(missing)}명")


In [ ]:
# 2. DEV 60문항 불러오기 (노트북 01 셀 7의 build_dev_set()과 같은 60문항)
#    질문은 저장소의 고정 평가셋 data/eval/questions_info_v2.json 을 그대로 쓴다 (지문으로 확인)
EXPECTED_SHA256 = "5677082a29dca568cb6222c092061ff0c27983c7cc3d8a03fcd0901736c61ad7"
QUESTIONS_PATH = REPO_DIR / "data" / "eval" / f"questions_{DATASET_VERSION}.json"

frozen = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
questions = frozen["questions"]
sha = hashlib.sha256(json.dumps(questions, ensure_ascii=False, sort_keys=True,
                                separators=(",", ":")).encode("utf-8")).hexdigest()
assert sha == EXPECTED_SHA256, f"고정 평가셋 파일 내용이 바뀌었습니다 (지문 {sha[:12]}). 손대지 말고 알려 주세요."

# 학습용 질문은 학습용 캐릭터에서만 근거를 찾는다 (노트북 01의 pool_for와 같다)
EVAL_CHARACTERS = set(frozen["eval_characters"])
TRAIN_POOL = [c for c in characters if c["name"] not in EVAL_CHARACTERS]


def few_shot_examples():
    """노트북 01과 같은 예시 3개. DEV에서 빼기 위해서만 쓴다 (rag_v4 예시는 core/rag_v4_examples.json)."""
    train = [q for q in questions if q["split"] == "train"]
    return [next(q for q in train if q["type"] == "lookup" and q["field"] == "element"),
            next(q for q in train if q["type"] == "filter" and 2 <= len(q["gold"]) <= 3),
            next(q for q in train if q["type"] == "unknown")]


def build_dev_set(n_per_field=10, n_filter=16, n_unknown=4):
    """노트북 01 셀 7과 같은 규칙: 학습용 질문에서 few-shot 예시를 빼고 해시 순서로 60개."""
    shot_ids = {ex["id"] for ex in few_shot_examples()}
    pool = sorted((q for q in questions if q["split"] == "train" and q["id"] not in shot_ids),
                  key=lambda q: hashlib.md5(("dev-" + q["id"]).encode("utf-8")).hexdigest())
    dev = []
    for field in ("element", "weapon", "rarity", "region"):
        dev += [q for q in pool if q["type"] == "lookup" and q["field"] == field][:n_per_field]
    dev += [q for q in pool if q["type"] == "filter"][:n_filter]
    dev += [q for q in pool if q["type"] == "unknown"][:n_unknown]
    return sorted(dev, key=lambda q: hashlib.md5(q["id"].encode("utf-8")).hexdigest())


DEV_QS = build_dev_set()
assert len(DEV_QS) == 60 and all(q["split"] == "train" for q in DEV_QS), "DEV 세트 이상"
print(f"고정 평가셋 사용: {QUESTIONS_PATH} (지문 {sha[:12]})")
print(f"DEV 세트 {len(DEV_QS)}문항: " + ", ".join(f"{t} {n}" for t, n in sorted(Counter(q['type'] for q in DEV_QS).items())))


In [ ]:
# 3. 채점: 답 정규화 + 집합 F1 + 모델 답변 파싱 (노트북 01 셀 3과 같다. 질문 분리 검사는 01에서 한다)
def canonical(text):
    """표기 차이만 맞춘다: '불 원소'->'불', '5'/'5성급'/'★5'->'5성', '몬드 출신'->'몬드'."""
    t = str(text).strip().strip("\"'「」").strip()
    t = re.sub(r"\s*(원소|출신|지역)$", "", t)
    m = re.fullmatch(r"[★☆]?\s*(\d)\s*(성급|성|star)?", t, re.I)
    if m:
        return f"{m.group(1)}성"
    return t.lower()


def set_f1(pred, gold):
    """정밀도/재현율/F1. 정답과 예측이 모두 비어 있으면 만점 (unknown 질문에서 '모른다'고 답한 경우)."""
    p, g = {canonical(x) for x in pred if str(x).strip()}, {canonical(x) for x in gold}
    tp, fp, fn = len(p & g), len(p - g), len(g - p)
    if not p and not g:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0, "tp": 0, "fp": 0, "fn": 0}
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}


def parse_list_answer(text):
    """모델 글에서 목록을 뽑는다. JSON 배열이면 그대로, 아니면 쉼표/줄바꿈으로 나눈다."""
    m = re.search(r"\[.*?\]", text, re.S)
    if m:
        try:
            return [str(x).strip() for x in json.loads(m.group(0))]
        except json.JSONDecodeError:
            pass
    parts = re.split(r"[,\n、]", text)
    return [re.sub(r"^[\-\*\d\.\s]+", "", p).strip() for p in parts if p.strip()]


def summarize(per_question):
    """문항별 결과 -> macro/micro 지표."""
    n = len(per_question)
    tp = sum(x["tp"] for x in per_question)
    fp = sum(x["fp"] for x in per_question)
    fn = sum(x["fn"] for x in per_question)
    micro_p = tp / (tp + fp) if tp + fp else 0.0
    micro_r = tp / (tp + fn) if tp + fn else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if micro_p + micro_r else 0.0
    return {"n": n,
            "macro_f1": round(sum(x["f1"] for x in per_question) / n, 4) if n else 0.0,
            "micro_precision": round(micro_p, 4), "micro_recall": round(micro_r, 4),
            "micro_f1": round(micro_f1, 4),
            "empty_answers": sum(1 for x in per_question if not x["pred"]),
            # 응답 시간 (예전 결과 파일에는 seconds가 없어서 0으로 본다)
            "avg_seconds": round(sum(x.get("seconds", 0) for x in per_question) / n, 3) if n else 0.0,
            "max_seconds": round(max((x.get("seconds", 0) for x in per_question), default=0), 3)}


def evaluate(answer_fn, qs, progress=True):
    """answer_fn(q) -> 모델 원문(문자열). 파싱과 채점은 여기서 한다. 원문도 저장한다."""
    iterator = qs
    if progress:
        try:
            from tqdm.auto import tqdm
            iterator = tqdm(qs, desc="질문 처리")
        except ImportError:
            pass
    per_question = []
    for q in iterator:
        started = time.perf_counter()
        raw = answer_fn(q)
        seconds = time.perf_counter() - started          # 질문별 응답 시간 (가이드 기본 3-2)
        pred = parse_list_answer(raw)
        s = set_f1(pred, q["gold"])
        per_question.append({"id": q["id"], "type": q["type"], "field": q["field"],
                             "question": q["question"], "gold": q["gold"], "pred": pred, "raw": raw,
                             "precision": round(s["precision"], 4), "recall": round(s["recall"], 4),
                             "f1": round(s["f1"], 4), "tp": s["tp"], "fp": s["fp"], "fn": s["fn"],
                             "seconds": round(seconds, 3)})
    return per_question, summarize(per_question)


def breakdown(per_question, key="type"):
    """유형(type) 또는 항목(field)별 지표."""
    groups = defaultdict(list)
    for x in per_question:
        groups[x[key]].append(x)
    return {k: summarize(v) for k, v in sorted(groups.items())}


def run_self_tests():
    assert canonical("불 원소") == "불" and canonical("5") == "5성" and canonical("★4") == "4성"
    assert canonical("5성급") == "5성" and canonical("몬드 출신") == "몬드"
    assert set_f1(["알베도"], ["알베도"])["f1"] == 1.0
    assert set_f1([], [])["f1"] == 1.0                       # unknown 질문에서 빈 답 = 정답
    assert set_f1(["바위"], [])["f1"] == 0.0                 # unknown 질문에서 지어낸 답 = 오답
    assert round(set_f1(["a", "b"], ["a", "c"])["f1"], 2) == 0.5
    assert parse_list_answer('답: ["불", "물"]') == ["불", "물"]
    assert parse_list_answer("불, 물") == ["불", "물"]
    assert parse_list_answer("[]") == []
    print("채점/파싱 테스트 통과")


run_self_tests()

In [ ]:
# 4. 근거 검색 + 프롬프트: 저장소 core/ 코드 그대로 (챗봇, 노트북 01의 rag_v4와 같은 프롬프트)
#    rag_v4 프롬프트가 노트북 01과 글자까지 같은지는 tests/test_prompt_v4.py가 확인한다
from core.answer import build_prompt   # noqa: E402

METHOD = "rag_v4"
ROUTER = Router(KB)


def evidence_for(q):
    """DEV는 학습용 질문이라 학습용 캐릭터에서만 근거를 찾는다."""
    # rag_v4는 예전 조건 검색(하나라도 만족, 상위 30명), rag_v5는 '조건 모두 만족' (core/retrieval.py)
    return core_retrieve(ROUTER.route(q["question"]), KB, TRAIN_POOL, filter_mode="all" if METHOD == "rag_v5" else "any")


def prompt_for(q):
    return build_prompt(q["question"], evidence_for(q), METHOD)


sample = next(q for q in DEV_QS if q["type"] == "lookup")
print(f"[{METHOD} 프롬프트 예시: 끝부분]\n..." + prompt_for(sample)[-300:])


---
# 모델 불러오기

**먼저**: 코랩 메뉴 `런타임 > 런타임 유형 변경 > T4 GPU`. 셀 5 출력에 `GPU 사용 가능: True`가 나와야 한다.
모델을 바꿀 때는 `런타임 > 세션 다시 시작` 후 셀 1부터 다시 실행한다 (이전 모델이 GPU 메모리에 남지 않게).


In [ ]:
# 5. 모델 불러오기: 아래 세 줄만 바꾼다 (사용 예는 맨 위 표)
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_TAG = "qwen2.5-7b"      # 결과 파일 이름에 붙는다: results/dev_rag_v4_{MODEL_TAG}.json
LOAD_4BIT = True             # 7B 이상은 True (T4 16GB에 fp16으로는 안 들어간다)

MAX_NEW_TOKENS = 160          # 노트북 01과 같게
import torch                                                          # noqa: E402
from transformers import AutoModelForCausalLM, AutoTokenizer          # noqa: E402

if LOAD_4BIT:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)


def load_model(model_id, load_4bit=False):
    """T4는 bf16이 안 돼서 fp16으로 불러온다. load_4bit=True면 4bit(nf4)로 불러온다."""
    kwargs = {"device_map": "auto"}
    if load_4bit:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16)
    try:
        return AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, **kwargs)
    except TypeError:   # transformers 버전에 따라 인자 이름이 다르다
        return AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, **kwargs)


print("GPU 사용 가능:", torch.cuda.is_available())
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = load_model(MODEL_ID, LOAD_4BIT)
GPU_MEMORY_GB = round(torch.cuda.memory_allocated() / 1024 ** 3, 2) if torch.cuda.is_available() else 0.0
print(f"불러옴: {MODEL_ID} ({'4bit' if LOAD_4BIT else 'fp16'}), GPU 메모리 {GPU_MEMORY_GB}GB")


def generate(prompt, max_new_tokens=MAX_NEW_TOKENS):
    """노트북 01 셀 6과 같다. enable_thinking=False는 Qwen3처럼 '생각 모드'가 있는 모델만 쓰고 나머지는 무시한다."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    kwargs = dict(max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    with torch.no_grad():
        try:
            out = model.generate(**inputs, **kwargs, stop_strings=["]"], tokenizer=tokenizer)
        except (TypeError, ValueError):   # 오래된 transformers는 stop_strings를 모른다
            out = model.generate(**inputs, **kwargs)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


q0 = next(q for q in DEV_QS if q["type"] == "filter")
start = time.time()
print(f"점검: {q0['question']} -> {generate(prompt_for(q0))!r} ({time.time() - start:.1f}초)")


In [ ]:
# 6. DEV 60문항 실행 -> 후처리 없음/있음 두 결과를 저장 (노트북 01의 결과 JSON과 같은 형식)
#    results/Model compare/dev_rag_v4_{MODEL_TAG}.json       : 후처리 없음
#    results/Model compare/dev_rag_v4_{MODEL_TAG}_post.json  : 같은 모델 원문에 후처리(core/answer.py의 clean_answer)만 적용
#    모델은 한 번만 돌린다. 후처리는 저장된 원문으로 다시 채점할 뿐이다
#    저장 위치: 드라이브 results/Model compare/ (기준 결과 dev_rag_v4*.json은 results/ 에 그대로 둔다)
from core.answer import POST_CUTOFF, clean_answer   # noqa: E402

COMPARE_DIR = RESULT_DIR / "Model compare"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ID = f"dev_{METHOD}_{MODEL_TAG}"
assert MODEL_TAG and EXPERIMENT_ID != f"dev_{METHOD}", "MODEL_TAG를 정해야 기존 dev_rag_v4.json을 덮어쓰지 않는다"


def save_result(experiment_id, method, per_question, seconds, extra_config=None, notes=""):
    result = {
        "experiment_id": experiment_id,
        "method": method,
        "model": MODEL_ID,
        "dataset_version": f"{DATASET_VERSION}-dev",
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "seconds": seconds,
        "limit": None,
        "config": {"code_version": CODE_VERSION, "max_new_tokens": MAX_NEW_TOKENS, "split": "dev",
                   "load_4bit": LOAD_4BIT, "gpu_memory_gb": GPU_MEMORY_GB, "notebook": "02_model_compare",
                   **(extra_config or {})},
        "metrics": summarize(per_question),
        "by_type": breakdown(per_question, "type"),
        "by_field": breakdown(per_question, "field"),
        "per_question": per_question,
        "notes": notes,
    }
    path = COMPARE_DIR / f"{experiment_id}.json"
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    m = result["metrics"]
    print(f"저장: {path}  (macro F1 = {m['macro_f1']}, 평균 {m['avg_seconds']}초)")
    return result


def postprocess_rows(per_question):
    """저장된 모델 원문(raw)에 후처리만 적용해 다시 채점한다. 근거는 모델이 받은 것과 같다."""
    q_by_id = {q["id"]: q for q in DEV_QS}
    rows = []
    for x in per_question:
        texts = [e["text"] for e in evidence_for(q_by_id[x["id"]]) if e["kind"] == "character"]
        pred = clean_answer(parse_list_answer(x["raw"]), texts, POST_CUTOFF)
        s = set_f1(pred, x["gold"])
        rows.append({**x, "pred": pred, "precision": round(s["precision"], 4), "recall": round(s["recall"], 4),
                     "f1": round(s["f1"], 4), "tp": s["tp"], "fp": s["fp"], "fn": s["fn"]})
    return rows


start = time.time()
per_question, _ = evaluate(lambda q: generate(prompt_for(q)), DEV_QS)
seconds = round(time.time() - start, 1)
save_result(EXPERIMENT_ID, METHOD, per_question, seconds,
            notes=f"모델 비교: {METHOD} 프롬프트 고정, 모델만 {MODEL_ID}")
post_rows = postprocess_rows(per_question)
save_result(f"{EXPERIMENT_ID}_post", f"{METHOD}+post", post_rows, seconds,
            extra_config={"postprocess": "clean_answer", "cutoff": POST_CUTOFF},
            notes=f"{EXPERIMENT_ID}의 모델 원문에 후처리만 적용. 모델은 다시 돌리지 않음")
changed = [(x["question"], x["pred"], y["pred"]) for x, y in zip(per_question, post_rows) if x["pred"] != y["pred"]]
print(f"후처리로 답이 바뀐 문항 {len(changed)}개")
for question, old, new in changed:
    print(f"- {question}\n    빠짐: {[p for p in old if p not in new]}\n    바뀜: {[p for p in new if p not in old]}")


In [ ]:
# 7. 모델별 비교표: results/Model compare/의 결과 + results/의 기준 결과 (후처리 없음/있음 둘 다)
#    기준 결과 dev_rag_v4.json, dev_rag_v4_post.json = 노트북 01에서 Qwen2.5-3B로 돌린 것
COMPARE_DIR = RESULT_DIR / "Model compare"
paths = sorted(COMPARE_DIR.glob(f"dev_{METHOD}_*.json"))
paths += [p for p in (RESULT_DIR / f"dev_{METHOD}.json", RESULT_DIR / f"dev_{METHOD}_post.json") if p.exists()]
rows = []
for p in paths:
    r = json.loads(p.read_text(encoding="utf-8"))
    t = r["by_type"]
    rows.append((r["experiment_id"], r["model"], "있음" if r["method"].endswith("+post") else "없음",
                 r["config"].get("load_4bit", False), r["metrics"]["macro_f1"],
                 t["lookup"]["macro_f1"], t["filter"]["macro_f1"], t["unknown"]["macro_f1"],
                 r["metrics"]["avg_seconds"], r["metrics"]["max_seconds"], r["config"].get("gpu_memory_gb", "-")))

print(f"{'실험':<32}{'모델':<40}{'후처리':>5}{'4bit':>6}{'macro F1':>9}{'lookup':>8}{'filter':>8}{'unknown':>8}"
      f"{'평균초':>7}{'최대초':>7}{'GPU GB':>7}")
for row in sorted(rows, key=lambda x: (x[1], x[2])):
    print(f"{row[0]:<32}{row[1]:<40}{row[2]:>5}{str(row[3]):>6}{row[4]:>9.3f}{row[5]:>8.3f}{row[6]:>8.3f}"
          f"{row[7]:>8.3f}{row[8]:>7.2f}{row[9]:>7.2f}{str(row[10]):>7}")